# v2 PRA — Can we model PRA better than the market?
# Regression task: predict PRA. Baseline = prop_line as direct predictor.
# Goal: beat baseline MAE/RMSE using rolling features + min/max line.
# Walk-forward evaluation: 2023-24 → 2024-25 → 2025-26.

## Cell 1 — Imports & config

In [ ]:
from pathlib import Path
import subprocess, sys, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error
import lightgbm as lgb

repo_root = Path(subprocess.check_output(["git", "rev-parse", "--show-toplevel"], text=True).strip())
sys.path.insert(0, str(repo_root))
from src.nba_rebounds_modeling.duckdb_s3_creds import connect_duckdb_s3

TARGET = "PRA"
MARKET = "player_points_rebounds_assists"
SEASONS = ["2023-24", "2024-25", "2025-26"]
SEASON_DATE_RANGES = {
    "2023-24": ("2023-10-01", "2024-06-30"),
    "2024-25": ("2024-10-01", "2025-06-30"),
    "2025-26": ("2025-10-01", "2026-06-30"),
}

def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def mae(y_true, y_pred):
    return float(mean_absolute_error(y_true, y_pred))

print(f"target: {TARGET}  market: {MARKET}")

## Cell 2 — Load data: game logs + per-bookmaker props → min/max line

In [ ]:
con = connect_duckdb_s3()

# --- Game logs: PTS, REB, AST, MIN ---
logs_frames = []
for season in SEASONS:
    print(f"  Loading logs {season}...", flush=True)
    q = f"""
        SELECT PLAYER_NAME,
               CAST(PTS AS DOUBLE) AS PTS,
               CAST(REB AS DOUBLE) AS REB,
               CAST(AST AS DOUBLE) AS AST,
               CAST(MIN AS DOUBLE) AS MIN,
               GAME_DATE
        FROM read_csv_auto('s3://nba-api-mt/player_game_logs/{season}/*.csv',
                           header=true, ignore_errors=true)
    """
    f = con.execute(q).df()
    f["season"] = season
    logs_frames.append(f)

logs = pd.concat(logs_frames, ignore_index=True)
logs = logs[logs["MIN"] > 0].copy()
logs["GAME_DATE"] = pd.to_datetime(logs["GAME_DATE"], format="mixed").dt.date
logs["player_key"] = logs["PLAYER_NAME"].str.lower().str.strip()
print(f"Logs: {len(logs):,} rows")

# --- Props: load all bookmakers, compute per-game line stats ---
props_frames = []
for season in SEASONS:
    start_date, end_date = SEASON_DATE_RANGES[season]
    print(f"  Loading props {season}...", flush=True)
    q = f"""
        SELECT player,
               CAST(prop_line AS DOUBLE) AS prop_line,
               CAST(over_odds AS DOUBLE)  AS over_odds,
               CAST(under_odds AS DOUBLE) AS under_odds,
               game_time
        FROM read_csv_auto('s3://the-odds-api-mt/nba/historical_player_props/{season}/*.csv',
                           header=true, ignore_errors=true)
        WHERE market = '{MARKET}'
          AND game_time >= '{start_date}'
          AND game_time <= '{end_date}'
    """
    f = con.execute(q).df()
    f["season"] = season
    props_frames.append(f)

props_raw = pd.concat(props_frames, ignore_index=True)
props_raw["game_time"] = pd.to_datetime(props_raw["game_time"], format="mixed")
props_raw["game_date"] = props_raw["game_time"].dt.date
props_raw["player_key"] = props_raw["player"].str.lower().str.strip()
print(f"Props raw: {len(props_raw):,} rows")

# Compute per-game line stats (min, max, median, spread, n_books)
# Each row in props_raw is one bookmaker quoting the line
line_stats = (
    props_raw
    .groupby(["player_key", "game_date", "season"], as_index=False)
    .agg(
        player=("player", "first"),
        median_line=("prop_line", "median"),
        min_line=("prop_line", "min"),
        max_line=("prop_line", "max"),
        over_odds=("over_odds", "median"),
        under_odds=("under_odds", "median"),
        n_books=("prop_line", "count"),
    )
)
line_stats["line_range"] = line_stats["max_line"] - line_stats["min_line"]
print(f"\nPer-game line stats: {len(line_stats):,} rows")
print("line_range distribution:")
print(line_stats["line_range"].value_counts().sort_index().head(10))

# --- Merge with logs ---
df = line_stats.merge(
    logs[["player_key", "GAME_DATE", "PLAYER_NAME", "PTS", "REB", "AST", "MIN", "season"]],
    left_on=["player_key", "game_date"],
    right_on=["player_key", "GAME_DATE"],
    how="inner",
    suffixes=("", "_log"),
)
if "season_log" in df.columns:
    df["season"] = df["season"].fillna(df["season_log"])
    df = df.drop(columns=["season_log"], errors="ignore")

df["PRA"] = df["PTS"] + df["REB"] + df["AST"]
before = len(df)
df = df[df["MIN"] >= 15].copy()
df["game_date"] = pd.to_datetime(df["game_date"])

print(f"\nAfter join + MIN>=15: {len(df):,} rows (dropped {before-len(df):,})")
print(df.groupby("season")["PLAYER_NAME"].count())